# Arrays

¿Qué será un array? Bueno esto es un poco básico pero no hay que saltárselo. Al lío:

# ¿Cómo se alacenan los arrays en memoria?
La idea principal de un array es simple:

*Un array almacena elementos en posiciones consecutivas de memoria*


* 1 integer = 32 bits o 64 bits (dependiendo del lenguaje)
* 1 celda de memoria = 1 byte = 8 bits
* 1 Integer = 32 bits = 4 bytes = 4 celdas de memoria (CONTIGUAS)
* Ya que los arrays siempre se almacenan en celdas contiguas


Aquí tendriamos que diferenciar lo que es un array en un lenguaje como C, a lo que es en un lenguaje como python, porque **python no tiene arrays primitivos al estilo de C**.

**En C un array es literalmente memoria, sin metadatos.** Por ejemplo `int a[5];` reserva 5x4=20 bytes contiguos; `a`es solo la dirección del primer byte. No guarda su longitud ni su capacidad, no comprueba límites y su dirección es estable toda su vida. Eso es un **array estático**.

Para estudiar los arrays en python **vamos a usar numpy**. ¿Por qué? por que guarda los valores en crudo y contiguos (como en C) y es de tamaño fijo.

In [1]:
# Setup
import sys, time, ctypes, gc
import numpy as np

# Visualizaciones de la lección (reciben numpy arrays):
#   ver_memoria · ver_init · ver_get · ver_set · ver_traverse · ver_copy
#   ver_insert · ver_delete · ver_duplicacion
from visualizations.array_viz import (
    ver_memoria, ver_init, ver_get, ver_set, ver_traverse,
    ver_copy, ver_insert, ver_delete, ver_duplicacion,
)

print("numpy", np.__version__)

numpy 2.5.0


## ¿Cómo se almacena un array en memoria?

Un `numpy.ndarray` de `int32` guarda cada entero en **4 bytes contiguos**. Se puede verificar leyendo su dirección real (con `.ctypes.data`), su `itemsize`y sus `strides`.

In [2]:
test_array = np.array([10, 20, 30, 40, 50], dtype=np.int32)

base = test_array.ctypes.data
print("Dirección base :", hex(base))
print("itemsize       :", test_array.itemsize, "bytes (= 4 celdas de 1 byte)")
print("strides        :", test_array.strides, "-> 4 bytes entre elementos = CONTIGUO")
print("nbytes (datos) :", test_array.nbytes, "bytes")
print()
for i in range(len(test_array)):
    print(f"  test_array[{i}] = {test_array[i]:>2}  ->  {hex(base + i*test_array.itemsize)}   (offset {i*test_array.itemsize} bytes)")

# Acceso O(1): dir(i) = base + i*itemsize, sin recorrer nada
i = 3
print(f"\nAcceso directo a[{i}]: base + {i}*{test_array.itemsize} =", hex(base + i*test_array.itemsize))

Dirección base : 0x1120a9ff0
itemsize       : 4 bytes (= 4 celdas de 1 byte)
strides        : (4,) -> 4 bytes entre elementos = CONTIGUO
nbytes (datos) : 20 bytes

  test_array[0] = 10  ->  0x1120a9ff0   (offset 0 bytes)
  test_array[1] = 20  ->  0x1120a9ff4   (offset 4 bytes)
  test_array[2] = 30  ->  0x1120a9ff8   (offset 8 bytes)
  test_array[3] = 40  ->  0x1120a9ffc   (offset 12 bytes)
  test_array[4] = 50  ->  0x1120aa000   (offset 16 bytes)

Acceso directo a[3]: base + 3*4 = 0x1120a9ffc


In [3]:
# Los bytes CRUDOS en RAM (little-endian en x86/ARM): el 10 es 0a 00 00 00
raw = test_array.tobytes()
print("bytes totales de datos:", len(raw), "\n")
for i in range(len(test_array)):
    chunk = raw[i*4:(i+1)*4]
    print(f"  test_array[{i}] = {test_array[i]:>2}  ->  {' '.join(f'{b:02x}' for b in chunk)}")
print("\n¿0x0a == 10?", 0x0a == 10)

bytes totales de datos: 20 

  test_array[0] = 10  ->  0a 00 00 00
  test_array[1] = 20  ->  14 00 00 00
  test_array[2] = 30  ->  1e 00 00 00
  test_array[3] = 40  ->  28 00 00 00
  test_array[4] = 50  ->  32 00 00 00

¿0x0a == 10? True


In [4]:
# Visualización: cada int ocupa 4 celdas contiguas (prueba dtype=np.int64 -> 8 celdas)
ver_memoria(test_array)

## Complejidad de sus operaciones

### GET - acceder a un elemento de la lista

* O(1) temporal 
* O(1) espacial 

Esto es tricky. Acceder a un elemento si es O(1) porque el acceso te lo da la fórmula que ya hemos visto (`hex(base + i*test_array.itemsize)`). Te cuesta lo mismo acceder a un array de 10, que de 10 millones. 

Pero si tienes que acceder a N elementos, te cuesta O(N), porque tienes que hacer esa operacion para cada uno de los elementos, independientemente del tamaño del array. Es decir, esto si que escala linealmente. Aunque segun la IA, esta operación ya no seria un GET,
sino un **N GET**, y eso tiene su propio nombre: **traverse**, que es el recorrer el array.

Respecto a la complejidad espacial; esta mide la memoria extra (auxiliar) que necesita la operacion *aparte* del array que ya existe. Y GET no reserva casi nada. Cuando hacemos `a[1]` solo usamos un hueco para guardar la direccion ya calculamos con `hex(base + i*test_array.itemsize)`.


In [5]:
print("\nGET (O(1)):");     ver_get(test_array, 3)


GET (O(1)):


GET [3] = 40 -> dir = base + 3*4 = 0x1120a9ff0 + 12 = 0x1120a9ffc   (acceso directo, O(1))


### SET -asignar un valor a un elemento de la lista 

* O(1) temporal 
* O(1) espacial 

Misma mecánica con SET. Asignar un valor a una posición ya existente (eg. `a[i] = v`) siempre cuesta lo mismo, acceder a la posición y copiar la nueva variable, da igual si el array tenga 10 o 10 millones de elementos. 

Lo mismo para la complejidad espacial, es O(1) porque sobreescribe en el sitio.

In [6]:
print("\nSET (O(1)):");     ver_set(test_array, 1, 99)


SET (O(1)):


SET [1] = 99 -> se sobreescribe la celda en su sitio, sin desplazar nada. O(1).


### Inicializar (Init) un array de tamaño N 

* O(N) temporal 
* O(N) espacial

Este es bastante claro. Necesito crear e inicializar el array. A mayor N, más tiempo de inicilizacion (tengo que acceder y copiar cada valor) y más espacio en memoria necesito.


In [7]:
print("INIT (O(N)):");      ver_init(np.zeros(5, dtype=np.int32))

INIT (O(N)):


Init -> reservar + escribir 5 celdas = O(N) tiempo y espacio.


### Traverse (recorrer)  

* O(N) temporal
* O(1) espacial 

Literalmente N accesos O(1), por tanto O(N) en tiempo. Pero O(1) en espacio, siempre que solo lea, o no haga ninguna operacion, la memoria extra que necesita es fija, no depende de N.

In [8]:
print("\nTRAVERSE (O(N)):"); ver_traverse(test_array, animar=True)     # ver_traverse(a, animar=True) para animación


TRAVERSE (O(N)):


TRAVERSE -> 5 visitas, 1 contador -> O(N) tiempo, O(1) espacio.


### Copiar 

* O(N) temporal 
* O(N) espacial

La operacion de copiar consiste en la lectura de cada uno de los elementos del array y su escritura en otro lugar. Ambas cosas crecen con N linealmente. Mientrar mayor sea N, más espacio adicional necesito y más tiempo necesito para la escritura de ese nuevo array.

In [12]:
print("\nCOPY (O(N)):");    ver_copy(test_array)


COPY (O(N)):


COPY -> se reserva un buffer nuevo y se copian 5 elementos. O(N) tiempo y espacio. ¿Misma dirección? False


### Inserción 

* O(N) temporal 
* O(1) / O(N) espacial

La inserción consiste en el desplazamiento a la derecha de los elementos del array. Para añadir el elemento `i` hay que agrandar el array desplazando todos los que están detrás una casilla a la derecha.

Respecto a la memoria, en caso de que se limiten a mover valores en un bloque que ya existe, solo sería O(1), pero si tuviese que recrear el array completo, sería O(N). Esto es fácil de entender. Si la operación de inserción lo que hace es crear el nuevo array en otro lugar de la memoria, tiene que desplazarlo entero (siendo por lo tanto O(N)), pero si el array tiene huecos disponibles, solo tendría que desplazarlos uno a uno, siendo O(N) en tiempo, pero solo O(1) es espacio, ya que requeriría una única posición de memoria adicional. 

In [10]:
print("\nINSERT (O(N)):");  ver_insert(test_array, 1, 99)


INSERT (O(N)):


insert(1, 99) -> 4 desplazamientos a la derecha (verde). Naranja = nuevo. O(N).


### Borrado 

* O(N) temporal 
* O(N) espacial

El borrado consiste en el desplazamiento del los elementos hacia la izquierda del array. Para quitar el elemento `i` hay que cerrar el hueco desplazando todos los que están detrás una castilla a la izquierda. 

Si borramos el último elemento sería O(1), ya que no habría nada que desplazar, pero en cualquier otro caso sería O(N).

In [11]:
print("\nDELETE (O(N)):");  ver_delete(test_array, 2)


DELETE (O(N)):


delete(2) -> 2 desplazamientos a la izquierda para tapar el hueco. O(N).
